# Deep Past Initiative – Machine Translation

## Takeaway

- When the first letter of a word is capitalized it implies the word is a personal name or a place name (i.e. proper noun). When the word is in ALL CAPS, that implies it is a Sumerian logogram and was written in place of the Akkadian syllabic spelling for scribal simplicity.

- Can use the published_text's transliteration as the cleaned version for training.

- Can use the normed words in Lexicon to swap for training.

## Facts

- The transliteration words len < 150, translation words len < 300


In [1]:
!pip install -q evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.7 MB/s eta 0:00:00


In [2]:
import os 
import gc
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from sentence_transformers import SentenceTransformer, util
import evaluate
import matplotlib.pyplot as plt
import sentencepiece as spm
from transformers import T5Tokenizer

2026-01-20 05:47:19.584764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768888039.780717      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768888039.834660      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768888040.330293      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768888040.330327      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768888040.330330      24 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    MODEL_NAME = "google/byt5-small" 
    
    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512
    
    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 100
    LEARNING_RATE = 2e-4
    OUTPUT_DIR = "./byt5-akkadian-model"

    PATIENCE = 10

    DRY_RUN = False

In [4]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

train_sent_df = pd.read_csv("/kaggle/input/dpc-sentence-alignment-oare-firstword/sentence_alignment.csv")
train_sent_df.rename(columns={
    "translit_sent": "transliteration",
    "translation_sent_train": "translation"
}, inplace=True)

train_df = pd.concat([train_df, train_sent_df[train_df.columns]])

print(f"Train Data: {len(train_df)} docs.")

Train Data: 2613 docs.


# Build Corpus

In [6]:
SRC_COL = "transliteration"
TGT_COL = "translation"

# Normalize whitespace only (do NOT aggressively normalize yet)
def clean(s):
    return " ".join(str(s).split())

with open("spm_corpus.txt", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        f.write(clean(row[SRC_COL]) + "\n")
        f.write(clean(row[TGT_COL]) + "\n")

# Sentencepiece Tokenizer

In [7]:
spm.SentencePieceTrainer.train(
    input="/kaggle/working/spm_corpus.txt",
    model_prefix="akkadian_spm",
    vocab_size=3000,
    model_type="unigram",         # better for low-resource
    character_coverage=1.0,
    user_defined_symbols=[
        "<DIV>", "<NUM>"
    ],
    bos_id=-1,
    eos_id=1,
    pad_id=0,
    unk_id=2
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/spm_corpus.txt
  input_format: 
  model_prefix: akkadian_spm
  model_type: UNIGRAM
  vocab_size: 3000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <DIV>
  user_defined_symbols: <NUM>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 2
  bos_id: -1
  eos_id: 1
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
 

# Load Tokenizer

In [8]:
tokenizer = T5Tokenizer(
    vocab_file="/kaggle/working/akkadian_spm.model",
    extra_ids=0,
    pad_token="<pad>",
    eos_token="</s>",
    unk_token="<unk>"
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


# T5 model for training

In [9]:
from transformers import T5Config, T5ForConditionalGeneration

config = T5Config(
    vocab_size=tokenizer.vocab_size,
    d_model=256,  # token embedding size; 512 is the median size, a bigger size leads to overfitting
    d_ff=1024,  # dimension of feed forward layer, rule of thumb = 4 * d_model
    num_layers=6,  # number of blocks in the encoder
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=tokenizer.pad_token_id,
)

model = T5ForConditionalGeneration(config)

# Create Batched Dataset

In [10]:
from datasets import Dataset

def preprocess(examples):
    prefix = "translate Akkadian to English: "

    inputs = [prefix+ex for ex in examples[SRC_COL]]
    targets = [ex for ex in examples[TGT_COL]]

    model_inputs = tokenizer(
        inputs,
        max_length=Config.MAX_LENGTH,
        truncation=True,
        text_target=targets
    )
    return model_inputs
    
dataset = Dataset.from_pandas(train_df)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)

train_dataset = split_dataset["train"].map(preprocess, batched=True)
val_dataset = split_dataset["test"].map(preprocess, batched=True)

Map:   0%|          | 0/2221 [00:00<?, ? examples/s]

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

# Training

In [11]:
from sacrebleu.metrics import CHRF
import numpy as np

# update the metrics as competition metrics collapse during training phase
chrf = CHRF(word_order=2)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sentence-level chrF (macro)
    sent_scores = [
        chrf.sentence_score(p.strip(), [l.strip()]).score
        for p, l in zip(decoded_preds, decoded_labels)
    ]

    return {
        "sentence_chrf": float(np.mean(sent_scores))
    }


In [12]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, GenerationConfig

# GenerationConfig(
#     max_length=512,
#     num_beams=4,
#     no_repeat_ngram_size=4,
#     repetition_penalty=1.3,
#     length_penalty=1.1,
#     early_stopping=True,
# )


EPOCH = 3 if Config.DRY_RUN else Config.EPOCHS
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,      # higher LR since training from scratch
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=EPOCH,
    fp16=True,
    logging_steps=100,
    report_to="none",
    # ===== EARLY STOPPING CONFIG =====
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    # predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    # early stopping
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=Config.PATIENCE)],

)

trainer.train()

/tmp/ipykernel_24/4217013617.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,5.808100,5.126265
2,5.042600,4.685025
3,4.659600,4.378757
4,4.360600,4.145002
5,4.216900,4.005267
6,4.033500,3.879131
7,3.872400,3.771306
8,3.765000,3.698696
9,3.625900,3.623045
10,3.567400,3.560547


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=10564, training_loss=3.0005320881587543, metrics={'train_runtime': 1584.9155, 'train_samples_per_second': 140.134, 'train_steps_per_second': 17.54, 'total_flos': 3287697679466496.0, 'train_loss': 3.0005320881587543, 'epoch': 38.0})

# Save

In [13]:
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")

Model saved to ./byt5-akkadian-model


In [14]:
# predictions = trainer.predict(val_dataset)
# labels = predictions.label_ids
# labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
# decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# decoded_labels = [[label.strip()] for label in decoded_labels]
# pred_ids = predictions.predictions

# if isinstance(pred_ids, tuple):
#     pred_ids = pred_ids[0]

# decoded_preds = tokenizer.batch_decode(
#     pred_ids,
#     skip_special_tokens=True,
#     clean_up_tokenization_spaces=True,
# )
# decoded_preds = [pred.strip() for pred in decoded_preds]

# Dry run Inference

In [15]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

MAX_LENGTH = 512
DEVICE = "cuda"

tokenizer_loaded = AutoTokenizer.from_pretrained(Config.OUTPUT_DIR)
model_loaded = AutoModelForSeq2SeqLM.from_pretrained(Config.OUTPUT_DIR).to("cuda")
model_loaded.eval()


class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = ["translate Akkadian to English: "+i for i in self.texts]
        self.translation = df['translation'].astype(str).tolist()
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        translation = self.translation[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "translation": translation
        }

test_df = train_df.sample(5)
test_dataset = InferenceDataset(test_df, tokenizer_loaded)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


print("Starting Inference...")
all_predictions = []
actuals = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        actuals.extend(batch['translation'])
  
        outputs = model_loaded.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            do_sample=False,
            # num_beams=4,
            # early_stopping=True,
            # top_k=30, 
            # top_p=0.95
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])

pred = pd.DataFrame(
    {
        "predictions": all_predictions,
        "actuals": actuals
    }
    
)
pred

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Starting Inference...


100%|██████████| 1/1 [00:04<00:00,  4.44s/it]


,predictions,actuals
0,To Ennam-Aššur and Ali-ahum from Damiq-pī-Aššu...,To Adida and Ali-ahum from Damiq-pī-Aššur: Our...
1,To Šalim-Aššur from Iddin-abum: Concerning the...,"From ... and Lamassī to Adida, Ababa and Ali-a..."
2,5 shekel per month per mina,"In the presence of En(n)ānum, of Lā-qēpum"
3,x + 5 minas of copper owed by Iddin-abum.,... fodder ... ... fodder ... ... 5 donkeys .....
4,"Say to Kuliya, thus says Ababaya: Here, as for...","There, for the sake of the safety of every min..."
